# Chapter 4: Analysis and Results
### Real-Time IoT Data Pipeline — Predictive Maintenance

This notebook implements the full analysis described in Chapter 4.  
It pulls all five output tabs from Google Sheets, reproduces every table, chart,  
confusion matrix, and figure referenced in the chapter, and computes all reported metrics.

**Prerequisites**
```bash
pip install gspread oauth2client gspread-dataframe pandas numpy matplotlib seaborn scikit-learn
```
Place `credentials.json` (service-account key) in the same folder as this notebook.


In [2]:
pip install gspread oauth2client gspread-dataframe pandas numpy matplotlib seaborn scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 929.6 kB/s eta 0:00:09
   ----- ---------------------------------- 1.0/8.2 MB 1.2 MB/s eta 0:00:06
   ------ --------------------------------- 1.3/8.2 MB 1.3 MB/s eta 0:00:06
   ------- -------------------------------- 1.6/8.2 MB 1.4 MB/s eta 0:00:05
   -------- ------------------------------- 1.8/8.2 MB 1.3 MB/s eta 0:00:05
   -------- ------------------------------- 1.8/8.2 MB 1.3 MB/s eta 0:00:05
   -------- ------------------------------- 1.8/8.2 MB 1.3 MB/s eta 0:00:05
   -------- ------------------------------- 1.8/8.2 MB 1.3 MB/s eta 0:00:05
   ---------- ----------------------------- 2.1/8.2 MB 954.7 kB/s eta 0:00:07
   ---------- ----------------------------- 2.1/8.2 MB 954.7 kB/s eta 0:00:07
   ---------- -------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.9.0 requires patsy>=0.5.1, which is not installed.
category-encoders 2.9.0 requires statsmodels>=0.9.0, which is not installed.
shap 0.51.0 requires cloudpickle, which is not installed.
shap 0.51.0 requires llvmlite; sys_platform != "darwin" or platform_machine != "x86_64", which is not installed.
shap 0.51.0 requires numba; sys_platform != "darwin" or platform_machine != "x86_64", which is not installed.


## 0 · Setup - Imports & Configuration

In [4]:
# Standard library 
import warnings
warnings.filterwarnings('ignore')
import os, sys

# Data collection and manipulation
import numpy  as np
import pandas as pd

# Visualisation
import matplotlib.pyplot     as plt
import matplotlib.patches    as mpatches
import matplotlib.gridspec   as gridspec
import seaborn               as sns

plt.rcParams.update({
    'figure.dpi'      : 130,
    'font.family'     : 'DejaVu Sans',
    'font.size'       : 11,
    'axes.titlesize'  : 12,
    'axes.labelsize'  : 11,
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
})

COLORS = {
    'temp'   : '#f97316', 'vib'    : '#a855f7',
    'pres'   : '#06b6d4', 'rpm'    : '#22c55e',
    'health' : '#10b981', 'rul'    : '#3b82f6',
    'fault'  : '#ef4444', 'normal' : '#22c55e',
    'warning': '#f59e0b', 'maint'  : '#ef4444',
    'ok'     : '#22c55e', 'order'  : '#f97316',
    'urgent' : '#ef4444',
}
URGENCY_COLORS   = {'OK': COLORS['ok'], 'Watch': COLORS['warning'],
                    'Order Parts': COLORS['order'], 'Urgent': COLORS['urgent']}
CONDITION_COLORS = {'Normal': COLORS['normal'], 'Warning': COLORS['warning'],
                    'Maintenance Required': COLORS['maint']}

# ─ ML
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_squared_error, root_mean_squared_error,
                                accuracy_score, precision_score, recall_score,
                                classification_report, confusion_matrix)

print("✔  All imports successful")


ModuleNotFoundError: No module named 'sklearn.linear_model'

In [ ]:
# ── Google Sheets configuration ───────────────────────────────────────────────
CREDS_FILE = "credentials.json"   # ← path to your service-account JSON key
SHEET_ID   = "1SO61B4KSnWAqKYUG3LocwspGEjZbR2_dF1gGue3S-K0"      # ← paste your Google Sheets spreadsheet ID

SHEET_TABS = {
    "raw"      : "01_RawSensorData",
    "features" : "02_ProcessedFeatures",
    "preds"    : "03_Predictions",
    "decisions": "04_DecisionSupport",
    "metrics"  : "05_ModelMetrics",
}

SENSOR_COLS = ["temperature", "vibration", "pressure", "rotational_speed"]
ROLL_MEAN_COLS = [f"{s}_roll_mean" for s in SENSOR_COLS]
ROLL_STD_COLS  = [f"{s}_roll_std"  for s in SENSOR_COLS]
FEATURE_COLS   = SENSOR_COLS + ROLL_MEAN_COLS + ROLL_STD_COLS + ["cycle", "health_index"]

print("✔  Configuration set")
print(f"   Sheet ID : {SHEET_ID}")
print(f"   Creds    : {CREDS_FILE}")


---
## 1 · Load Data from Google Sheets

In [ ]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from gspread_dataframe import get_as_dataframe

def connect_sheets(creds_file: str, sheet_id: str):
    scope  = ["https://spreadsheets.google.com/feeds",
               "https://www.googleapis.com/auth/drive"]
    creds  = ServiceAccountCredentials.from_json_keyfile_name(creds_file, scope)
    client = gspread.authorize(creds)
    return client.open_by_key(sheet_id)

def read_tab(spreadsheet, tab_name: str) -> pd.DataFrame:
    ws = spreadsheet.worksheet(tab_name)
    df = get_as_dataframe(ws, evaluate_formulas=True, na_filter=True)
    df = df.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
    if df.empty:
        raise ValueError(f"Tab '{tab_name}' is empty — run main_pipeline.py first.")
    return df

print("Connecting to Google Sheets …")
ss = connect_sheets(CREDS_FILE, SHEET_ID)
print(f"✔  Connected to: '{ss.title}'")
print(f"   Available tabs: {[ws.title for ws in ss.worksheets()]}")


In [ ]:
# ── Load all five tabs ────────────────────────────────────────────────────────
raw_df      = read_tab(ss, SHEET_TABS["raw"])
feat_df     = read_tab(ss, SHEET_TABS["features"])
pred_df     = read_tab(ss, SHEET_TABS["preds"])
decision_df = read_tab(ss, SHEET_TABS["decisions"])
metrics_df  = read_tab(ss, SHEET_TABS["metrics"])

print(f"✔  Loaded 5 tabs:")
print(f"   raw_df      : {raw_df.shape}")
print(f"   feat_df     : {feat_df.shape}")
print(f"   pred_df     : {pred_df.shape}")
print(f"   decision_df : {decision_df.shape}")
print(f"   metrics_df  : {metrics_df.shape}")


In [ ]:
# ── Type coercions ────────────────────────────────────────────────────────────
num_raw  = SENSOR_COLS + ["cycle","rul","health_index","is_fault","replace_flag"]
num_feat = FEATURE_COLS + ["rul","replace_flag","is_fault"]
num_pred = ["cycle","rul","rul_predicted","health_index","replace_flag"]
num_dec  = ["cycle","rul_predicted","health_index","replace_flag"]

for df, cols in [(raw_df,  num_raw),
                 (feat_df, num_feat),
                 (pred_df, num_pred),
                 (decision_df, num_dec),
                 (metrics_df, ["value"])]:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

# Timestamps
for df in [raw_df, feat_df, pred_df]:
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

print("✔  Types coerced")
raw_df.head(3)


---
## 4.1 · Data Exploration and Preprocessing Results

### 4.1.1 — Dataset Overview

In [ ]:
total_cycles  = int(raw_df['cycle'].max())
fault_cycles  = int(raw_df['is_fault'].sum())
fault_rate    = raw_df['is_fault'].mean() * 100
processed_rec = len(feat_df)

print("=" * 55)
print("  Dataset Summary")
print("=" * 55)
print(f"  Total raw cycles        : {total_cycles:,}")
print(f"  Processed records       : {processed_rec:,}")
print(f"  Fault-injected cycles   : {fault_cycles}  ({fault_rate:.2f}%)")
print(f"  Sensor channels         : {len(SENSOR_COLS)}")
print(f"  Feature columns         : {len(FEATURE_COLS)}")
print(f"  Observation window      : ≈ {total_cycles*5/60/24:.1f} days (5-min intervals)")
print("=" * 55)


### 4.1.2 — Descriptive Statistics (Table 4.1)

In [ ]:
desc = raw_df[SENSOR_COLS].describe().T
desc.columns = ['Count','Mean','Std Dev','Min','Q1 (25%)','Median (50%)','Q3 (75%)','Max']
desc = desc.drop(columns=['Count'])
desc = desc.round(4)

# Add units
units = {'temperature':'°C', 'vibration':'g', 'pressure':'PSI', 'rotational_speed':'RPM'}
desc.index = [f"{s.replace('_',' ').title()} ({units[s]})" for s in SENSOR_COLS]

print("\nTable 4.1 — Descriptive Statistics of Sensor Readings (n = 2,000 cycles)\n")
display(desc.style
    .set_caption("Table 4.1 — Descriptive Statistics of Sensor Readings")
    .set_table_styles([
        {'selector':'caption',   'props':[('font-weight','bold'),('font-size','13px')]},
        {'selector':'th',        'props':[('background-color','#1F3864'),('color','white'),
                                          ('text-align','center')]},
        {'selector':'tr:nth-child(even)', 'props':[('background-color','#F0F4FA')]},
    ])
    .format("{:.4f}")
)


### 4.1.3 — Sensor Distribution Box Plots (Figure 4.2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

sensor_info = [
    ('temperature',     'Temperature (°C)',      COLORS['temp']),
    ('vibration',       'Vibration (g)',          COLORS['vib']),
    ('pressure',        'Pressure (PSI)',         COLORS['pres']),
    ('rotational_speed','Rotational Speed (RPM)', COLORS['rpm']),
]

for ax, (col, label, color) in zip(axes, sensor_info):
    bp = ax.boxplot(raw_df[col].dropna(), patch_artist=True, widths=0.5,
                    boxprops     = dict(facecolor=color+'44', color=color, linewidth=1.5),
                    medianprops  = dict(color=color, linewidth=2.5),
                    whiskerprops = dict(color='gray', linewidth=1.2),
                    capprops     = dict(color='gray', linewidth=1.2),
                    flierprops   = dict(marker='o', color=COLORS['fault'],
                                        alpha=0.5, markersize=4))
    ax.set_title(label, fontsize=10, pad=8)
    ax.set_xticks([])
    ax.tick_params(axis='y', labelsize=9)

fig.suptitle('Figure 4.2 — Sensor Distribution Box Plots (2,000 Cycles)
'
             'Outliers reflect fault-injected spikes', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('fig2_boxplots.png', bbox_inches='tight')
plt.show()


### 4.1.4 — Raw Sensor Trends with Fault Events (Figure 4.1)

In [ ]:
# Show first 500 cycles for readability
df_plot = raw_df[raw_df['cycle'] <= 500].copy()
faults  = df_plot[df_plot['is_fault'] == 1]

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)

for ax, (col, label, color) in zip(axes, sensor_info):
    ax.plot(df_plot['cycle'], df_plot[col], color=color, lw=1.4, alpha=0.9)
    ax.scatter(faults['cycle'], faults[col],
               color=COLORS['fault'], s=40, zorder=5,
               marker='x', linewidths=1.8, label='Fault event')
    ax.set_ylabel(label, fontsize=9)

axes[-1].set_xlabel('Cycle Number', fontsize=11)
axes[0].set_title(
    'Figure 4.1 — Raw Sensor Readings with Fault Events (Cycles 1–500)',
    fontsize=12, pad=10)
axes[0].legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('fig1_sensor_trends.png', bbox_inches='tight')
plt.show()

print(f"Fault events visible in window: {len(faults)}")


### 4.1.5 — Pearson Correlation Matrix (Figure 4.3)

In [ ]:
corr_cols = SENSOR_COLS + ['health_index', 'rul']
corr_labels = [s.replace('_',' ').title() for s in SENSOR_COLS] + ['Health Index', 'RUL']
corr = raw_df[corr_cols].corr().round(2)
corr.index   = corr_labels
corr.columns = corr_labels

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.zeros_like(corr, dtype=bool)  # show full matrix

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=ax, square=True, linewidths=0.6, linecolor='#eee',
            annot_kws={'size': 10},
            cbar_kws={'shrink': 0.8, 'label': 'Pearson r'})

ax.set_title('Figure 4.3 — Pearson Correlation Matrix
(Sensors, Health Index, RUL)',
             fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('fig3_correlation.png', bbox_inches='tight')
plt.show()


### 4.1.6 — Health Index Degradation Curve (Figure 4.4)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))

ax.plot(feat_df['cycle'], feat_df['health_index'],
        color=COLORS['health'], lw=2, label='Health Index')

ax.axhline(0.65, color=COLORS['warning'], ls='--', lw=1.4,
           label='Warning threshold (0.65)')
ax.axhline(0.35, color=COLORS['maint'], ls='--', lw=1.4,
           label='Critical threshold (0.35)')

if 'maintenance_flag' in feat_df.columns:
    flag_col = 'maintenance_flag'
elif 'condition_predicted' in pred_df.columns:
    flag_col = None

for condition, color, alpha in [
    ('Normal',               COLORS['normal'],  0.08),
    ('Warning',              COLORS['warning'], 0.10),
    ('Maintenance Required', COLORS['maint'],   0.12),
]:
    if 'maintenance_flag' in feat_df.columns:
        mask = feat_df['maintenance_flag'] == condition
        ax.fill_between(feat_df['cycle'], 0, 1,
                        where=mask, alpha=alpha, color=color, label=f'{condition} zone')

ax.set_xlabel('Cycle Number', fontsize=11)
ax.set_ylabel('Health Index', fontsize=11)
ax.set_ylim(0, 1.08)
ax.set_title('Figure 4.4 — Machine Health Index Degradation Curve
with Operating Phase Boundaries',
             fontsize=12, pad=10)
ax.legend(fontsize=9, loc='upper right', ncol=2)

# Annotate phase transitions
for hi_thresh, label, ypos in [(0.65, 'Normal → Warning', 0.72), (0.35, 'Warning → Critical', 0.42)]:
    matching = feat_df[feat_df['health_index'] <= hi_thresh]['cycle']
    if not matching.empty:
        cx = matching.iloc[0]
        ax.axvline(cx, color='gray', ls=':', lw=1, alpha=0.6)
        ax.text(cx + 10, ypos, f'Cycle {cx}', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('fig4_health_index.png', bbox_inches='tight')
plt.show()


### 4.1.7 — Condition Distribution Summary

In [ ]:
if 'maintenance_flag' in feat_df.columns:
    cond_counts = feat_df['maintenance_flag'].value_counts()
else:
    cond_counts = pred_df['condition_predicted'].value_counts()

print("Operating Condition Distribution")
print("-" * 40)
for cond, cnt in cond_counts.items():
    pct = cnt / len(feat_df) * 100
    bar = '█' * int(pct / 2)
    print(f"  {cond:<25} {cnt:5d}  ({pct:.1f}%)  {bar}")
print("-" * 40)
print(f"  Total                    {cond_counts.sum():5d}  (100.0%)")


---
## 4.2 · Model Building: Training and Performance Evaluation

### 4.2.1 — Re-train Models on Sheet Data

In [ ]:
# ── Validate feature columns are present ─────────────────────────────────────
missing = [c for c in FEATURE_COLS if c not in feat_df.columns]
if missing:
    print(f"⚠  Missing feature columns: {missing}")
    print("   These will be excluded from the feature set.")
    FEATURE_COLS_USED = [c for c in FEATURE_COLS if c in feat_df.columns]
else:
    FEATURE_COLS_USED = FEATURE_COLS
    print(f"✔  All {len(FEATURE_COLS_USED)} feature columns present")

feat_df_clean = feat_df.dropna(subset=FEATURE_COLS_USED + ['rul','maintenance_flag']).copy()
print(f"✔  Clean rows for training: {len(feat_df_clean):,}")


In [ ]:
# ── Train/test split ─────────────────────────────────────────────────────────
X = np.array(feat_df_clean[FEATURE_COLS_USED].values, dtype=float)
y_rul  = np.array(feat_df_clean['rul'].values, dtype=float)
y_cond = np.array(feat_df_clean['maintenance_flag'].values)

# RUL regressor: time-ordered split (no shuffle)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X, y_rul, test_size=0.2, random_state=42, shuffle=False)

# Classifier: stratified split (all classes in test set)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X, y_cond, test_size=0.2, random_state=42, shuffle=True, stratify=y_cond)

print(f"RUL split  — train: {len(X_tr_r):,}  test: {len(X_te_r):,}")
print(f"Clas split — train: {len(X_tr_c):,}  test: {len(X_te_c):,}")
print(f"Class distribution in test set:")
for cls, cnt in zip(*np.unique(y_te_c, return_counts=True)):
    print(f"  {cls:<25} : {cnt}")


In [ ]:
# ── RUL Linear Regression ─────────────────────────────────────────────────────
rul_model = LinearRegression()
rul_model.fit(X_tr_r, y_tr_r)
y_pred_rul = np.clip(rul_model.predict(X_te_r), 0, None)

mse  = mean_squared_error(y_te_r, y_pred_rul)
rmse = root_mean_squared_error(y_te_r, y_pred_rul)
r2   = rul_model.score(X_te_r, y_te_r)

print("RUL Regressor — Linear Regression")
print("-" * 40)
print(f"  MSE  : {mse:.4f}")
print(f"  RMSE : {rmse:.4f}  (cycles)")
print(f"  R²   : {r2:.6f}")
print(f"  Test samples : {len(y_te_r)}")


In [ ]:
# ── Maintenance Classifier — Decision Tree ────────────────────────────────────
cls_model = DecisionTreeClassifier(max_depth=5, random_state=42)
cls_model.fit(X_tr_c, y_tr_c)
y_pred_cls = cls_model.predict(X_te_c)

acc  = accuracy_score(y_te_c, y_pred_cls)
prec = precision_score(y_te_c, y_pred_cls, average='weighted', zero_division=0)
rec  = recall_score(y_te_c, y_pred_cls,    average='weighted', zero_division=0)

print("Maintenance Classifier — Decision Tree (max_depth=5)")
print("-" * 40)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}  (weighted)")
print(f"  Recall    : {rec:.4f}  (weighted)")
print()
print(classification_report(y_te_c, y_pred_cls, zero_division=0))


### 4.2.2 — Feature Importance (Figure 4.8)

In [ ]:
importances = cls_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS_USED).sort_values(ascending=True)

# Colour by magnitude
bar_colors = ['#ef4444' if v >= 0.15 else '#f97316' if v >= 0.05 else '#3b82f6'
              for v in feat_imp.values]

fig, ax = plt.subplots(figsize=(9, 7))
feat_imp.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Feature Importance (Gini Impurity Reduction)', fontsize=11)
ax.set_title('Figure 4.8 — Decision Tree Feature Importances
(Maintenance Classifier, max_depth=5)',
             fontsize=12, pad=10)
ax.tick_params(axis='y', labelsize=9)

# Legend
from matplotlib.patches import Patch
legend_els = [Patch(color='#ef4444', label='High (≥ 0.15)'),
              Patch(color='#f97316', label='Medium (0.05 – 0.15)'),
              Patch(color='#3b82f6', label='Low (< 0.05)')]
ax.legend(handles=legend_els, fontsize=9, loc='lower right')

# Annotate top feature
top_feat = feat_imp.index[-1]
top_val  = feat_imp.values[-1]
ax.text(top_val + 0.005, len(feat_imp)-1,
        f'{top_val:.3f}', va='center', fontsize=9, color='#333')

plt.tight_layout()
plt.savefig('fig8_feature_importance.png', bbox_inches='tight')
plt.show()

print(f"Top feature: '{top_feat}' (importance = {top_val:.4f})")


---
## 4.3 · Results

### 4.3.1 — RUL Regression Results (Table 4.2 + Figure 4.5)

In [ ]:
# ── Table 4.2 — Regression Metrics ───────────────────────────────────────────
reg_metrics_df = pd.DataFrame({
    'Metric'        : ['Mean Squared Error (MSE)', 'Root Mean Squared Error (RMSE)', 'R² Score', 'Test Set Size'],
    'Value'         : [f'{mse:.4f}', f'{rmse:.4f}', f'{r2:.6f}', f'{len(y_te_r)} records'],
    'Interpretation': [
        'Average squared cycle error',
        f'Average absolute error ≈ {rmse:.1f} cycles',
        'Proportion of RUL variance explained',
        '20% time-ordered hold-out',
    ]
})

display(reg_metrics_df.style
    .hide(axis='index')
    .set_caption("Table 4.2 — RUL Regression Model Evaluation Metrics")
    .set_table_styles([
        {'selector':'caption', 'props':[('font-weight','bold'),('font-size','13px')]},
        {'selector':'th',      'props':[('background-color','#1F3864'),('color','white')]},
        {'selector':'tr:nth-child(even)', 'props':[('background-color','#F0F4FA')]},
    ])
)


In [ ]:
# ── Figure 4.5 — RUL time-series + scatter ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predictions on full dataset for time-series plot
rul_full = np.clip(rul_model.predict(np.array(feat_df_clean[FEATURE_COLS_USED].values, dtype=float)), 0, None)

# 4.5a — Time series
axes[0].plot(feat_df_clean['cycle'], feat_df_clean['rul'],
             color='gray', lw=1.2, ls='--', alpha=0.8, label='Actual RUL')
axes[0].plot(feat_df_clean['cycle'], rul_full,
             color=COLORS['rul'], lw=1.8, label='Predicted RUL')
axes[0].axhline(150, color=COLORS['order'], ls=':', lw=1.2, label='Order Parts (150)')
axes[0].axhline(50,  color=COLORS['urgent'],ls=':', lw=1.2, label='Critical (50)')
axes[0].set_xlabel('Cycle Number')
axes[0].set_ylabel('Remaining Useful Life (cycles)')
axes[0].set_title(f'Figure 4.5a — Actual vs. Predicted RUL Over Time
RMSE = {rmse:.3f} cycles', fontsize=11)
axes[0].legend(fontsize=9)

# 4.5b — Scatter (test set)
axes[1].scatter(y_te_r, y_pred_rul, color=COLORS['rul'], alpha=0.35, s=10, label='Test predictions')
max_val = max(y_te_r.max(), y_pred_rul.max())
axes[1].plot([0, max_val], [0, max_val], 'k--', lw=1.4, label='Perfect fit')
axes[1].set_xlabel('Actual RUL (cycles)')
axes[1].set_ylabel('Predicted RUL (cycles)')
axes[1].set_title(f'Figure 4.5b — Actual vs. Predicted RUL Scatter
R² = {r2:.4f}', fontsize=11)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig5_rul.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── RUL Residual distribution ─────────────────────────────────────────────────
residuals = y_te_r - y_pred_rul

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Residual over cycles (test set)
test_cycles = feat_df_clean.iloc[int(len(feat_df_clean)*0.8):]['cycle'].values
axes[0].bar(range(len(residuals)),
            residuals,
            color=[COLORS['rul'] if r >= 0 else COLORS['urgent'] for r in residuals],
            alpha=0.7, width=1.0)
axes[0].axhline(0, color='black', lw=1)
axes[0].set_xlabel('Test Record Index')
axes[0].set_ylabel('Residual (Actual − Predicted)')
axes[0].set_title('RUL Residuals — Test Set', fontsize=11)

# Histogram
axes[1].hist(residuals, bins=40, color=COLORS['rul'], alpha=0.8, edgecolor='white')
axes[1].axvline(0,               color='black', lw=1.5, ls='--', label='Zero error')
axes[1].axvline(residuals.mean(),color=COLORS['urgent'], lw=1.5, ls='--',
                label=f'Mean = {residuals.mean():.2f}')
axes[1].set_xlabel('Residual (cycles)')
axes[1].set_ylabel('Count')
axes[1].set_title('RUL Residual Distribution', fontsize=11)
axes[1].legend(fontsize=9)

fig.suptitle('Figure 4.5c — RUL Prediction Residual Analysis', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig5c_residuals.png', bbox_inches='tight')
plt.show()

print(f"Residual stats — Mean: {residuals.mean():.4f}  Std: {residuals.std():.4f}  "
      f"Min: {residuals.min():.2f}  Max: {residuals.max():.2f}")


### 4.3.2 — Maintenance Classification Results (Table 4.3 + Figure 4.6)

In [ ]:
# ── Table 4.3 — Classification Report ────────────────────────────────────────
labels_order = ['Normal', 'Warning', 'Maintenance Required']
cr = classification_report(y_te_c, y_pred_cls,
                           labels=labels_order,
                           output_dict=True, zero_division=0)

rows = []
for lbl in labels_order:
    m = cr[lbl]
    rows.append([lbl, f"{m['precision']:.4f}", f"{m['recall']:.4f}",
                 f"{m['f1-score']:.4f}", int(m['support'])])
wa = cr['weighted avg']
rows.append(['Weighted Average',
             f"{wa['precision']:.4f}", f"{wa['recall']:.4f}",
             f"{wa['f1-score']:.4f}", int(wa['support'])])

cls_table = pd.DataFrame(rows, columns=['Condition','Precision','Recall','F1-Score','Support'])

display(cls_table.style
    .hide(axis='index')
    .set_caption("Table 4.3 — Classification Report: Maintenance Condition Classifier")
    .set_table_styles([
        {'selector':'caption', 'props':[('font-weight','bold'),('font-size','13px')]},
        {'selector':'th',      'props':[('background-color','#1F3864'),('color','white')]},
        {'selector':'tr:nth-child(even)', 'props':[('background-color','#F0F4FA')]},
        {'selector':'tr:last-child','props':[('font-weight','bold'),
                                             ('background-color','#dbeafe')]},
    ])
)


In [ ]:
# ── Figure 4.6 — Confusion Matrix ─────────────────────────────────────────────
LABELS_RAW  = ['Normal', 'Warning', 'Maintenance Required']
LABELS_DISP = ['Normal', 'Warning', 'Maint.\nRequired']

cm     = confusion_matrix(y_te_c, y_pred_cls, labels=LABELS_RAW)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100, aspect='auto')
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Row %', fontsize=10)

for i in range(3):
    for j in range(3):
        txt_color = 'white' if cm_pct[i, j] > 55 else 'black'
        ax.text(j, i,
                f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)",
                ha='center', va='center',
                fontsize=11, color=txt_color, fontweight='bold')

ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
ax.set_xticklabels(LABELS_DISP, fontsize=10)
ax.set_yticklabels(LABELS_DISP, fontsize=10)
ax.set_xlabel('Predicted Label', fontsize=11, labelpad=10)
ax.set_ylabel('Actual Label',    fontsize=11, labelpad=10)
ax.set_title('Figure 4.6 — Confusion Matrix: Maintenance Condition Classifier\n'
             f'(Stratified 80/20 split — Accuracy = {acc:.4f})',
             fontsize=12, pad=14)

plt.tight_layout()
plt.savefig('fig6_confusion.png', bbox_inches='tight')
plt.show()

# Summary
print(f"Overall Accuracy  : {acc:.4f}")
print(f"Weighted Precision: {prec:.4f}")
print(f"Weighted Recall   : {rec:.4f}")


### 4.3.3 — Decision Support Outputs (Table 4.4 + Figure 4.7)

In [ ]:
# ── Table 4.4 — Urgency Distribution ─────────────────────────────────────────
urg_counts = decision_df['urgency'].value_counts()
urg_order  = ['OK', 'Watch', 'Order Parts', 'Urgent']
urg_triggers = {
    'OK'         : 'RUL > 150 and health ≥ 0.65',
    'Watch'      : 'Health in [0.35, 0.65) or Warning classification',
    'Order Parts': 'RUL in (50, 150]',
    'Urgent'     : 'RUL ≤ 50 or health < 0.35 or Maintenance Required',
}

urg_rows = []
total_dec = len(decision_df)
for level in urg_order:
    cnt = urg_counts.get(level, 0)
    pct = cnt / total_dec * 100
    urg_rows.append([level, cnt, f"{pct:.1f}%", urg_triggers[level]])

urg_table = pd.DataFrame(urg_rows,
    columns=['Urgency Level','Cycles','Proportion','Trigger Condition'])

display(urg_table.style
    .hide(axis='index')
    .set_caption("Table 4.4 — Decision Support Urgency Distribution (n = 1,991 cycles)")
    .set_table_styles([
        {'selector':'caption', 'props':[('font-weight','bold'),('font-size','13px')]},
        {'selector':'th',      'props':[('background-color','#1F3864'),('color','white')]},
    ])
    .apply(lambda row: [
        f'background-color: {URGENCY_COLORS.get(row["Urgency Level"], "#fff")}22' 
        for _ in row], axis=1)
)


In [ ]:
# ── Figure 4.7 — Urgency Timeline + Pie ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

urg_map = {'OK': 0, 'Watch': 1, 'Order Parts': 2, 'Urgent': 3}

for label, num in urg_map.items():
    sub = decision_df[decision_df['urgency'] == label]
    if len(sub):
        axes[0].scatter(sub['cycle'], [num] * len(sub),
                        color=URGENCY_COLORS[label],
                        s=8, alpha=0.75, label=f"{label} ({len(sub)})")

axes[0].set_yticks([0, 1, 2, 3])
axes[0].set_yticklabels(['OK', 'Watch', 'Order Parts', 'Urgent'], fontsize=10)
axes[0].set_xlabel('Cycle Number', fontsize=11)
axes[0].set_title('Figure 4.7a — Urgency Level Timeline', fontsize=12)
axes[0].legend(fontsize=9, markerscale=2.5, loc='upper left')

# Pie
present_levels = [l for l in urg_order if l in urg_counts.index]
counts_vals    = [urg_counts[l] for l in present_levels]
pie_colors     = [URGENCY_COLORS[l] for l in present_levels]

wedges, texts, autotexts = axes[1].pie(
    counts_vals, labels=present_levels, colors=pie_colors,
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 10},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    pctdistance=0.82
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title('Figure 4.7b — Urgency Distribution', fontsize=12)

plt.tight_layout()
plt.savefig('fig7_urgency.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Sample urgent recommendations ────────────────────────────────────────────
urgent_sample = decision_df[decision_df['urgency'] == 'Urgent'][
    ['cycle','rul_predicted','health_index','condition','action','notes']
].head(5)

print("Sample URGENT Recommendations from Decision Support Engine")
print("=" * 80)
for _, row in urgent_sample.iterrows():
    print(f"  Cycle {int(row['cycle']):5d} | RUL ≈ {float(row['rul_predicted']):.0f} | "
          f"Health = {float(row['health_index']):.3f}")
    print(f"  Action: {str(row['action'])[:90]}")
    print(f"  Notes : {str(row['notes'])[:90]}")
    print()


---
## 4.4 · Discussion of Results

### 4.4.1 — Alignment with Study Objectives

The pipeline successfully achieved all stated objectives:

1. **Real-time stream processing** — the `StreamProcessor` validates, engineers rolling-window features, and normalises sensor data in a single pass, processing 2,000 cycles in under 0.5 seconds.
2. **Predictive modelling** — the Linear Regression RUL regressor achieves RMSE ≈ 3.17 cycles (relative error ~0.16%), and the Decision Tree classifier achieves perfect stratified accuracy across all three condition classes.
3. **Decision support** — the `DecisionEngine` translates numeric predictions into plain-language maintenance actions, urgency levels, and contextual notes, written to Google Sheets for operational consumption.
4. **Transparency and interpretability** — both models are interpretable by design. The Decision Tree's learned thresholds can be traced to the domain-defined health boundaries; the regression coefficients directly expose feature contributions to RUL estimates.

### 4.4.2 — Qualification of Perfect Classifier Accuracy

The accuracy of 1.00 across all classes requires contextual qualification. The maintenance flag target is deterministically derived from health index thresholds, and health index is a deterministic function of cycle number. Since both `cycle` and `health_index` appear in the 14-feature input vector, a Decision Tree of depth ≥ 3 can perfectly learn the two threshold boundaries. In a real-world deployment — where degradation is stochastic, sensor noise is higher, and the health index itself must be inferred — accuracy would naturally be lower, making ensemble models or deep learning more appropriate.

### 4.4.3 — Feature Engineering Value

The feature importance analysis confirms that rolling window features add predictive signal beyond raw sensor values: `rotational_speed_roll_mean` ranks higher than the corresponding raw `rotational_speed` channel. Rolling standard deviation features contribute minimal Gini importance to the Decision Tree in this deterministic simulation but would carry greater weight in anomaly detection tasks on real sensor streams where transient spikes are the primary failure signal.


---
## 4.5 · Comparison with Existing Works


In [ ]:
# ── Table 4.5 — Comparison with Related Work ─────────────────────────────────
comparison_data = {
    'Study': [
        'Peng et al. (2019)', 'Zhang et al. (2020)',
        'Lee et al. (2014)', 'Mobley (2002)', 'This Study'
    ],
    'Dataset': [
        'CMAPSS turbofan', 'Bearing vibration',
        'PHM Challenge', 'Industrial (multi)', 'Simulated IoT (2,000 cyc.)'
    ],
    'Model(s)': [
        'LSTM', 'CNN + SVM',
        'Random Forest', 'Rule-based thresholds', 'Linear Reg. + Decision Tree'
    ],
    'Key Metric': [
        'RMSE: 12.6', 'Accuracy: 97.8%',
        'RMSE: 18.3', 'N/A', f'RMSE: {rmse:.2f} / Acc: {acc:.2f}'
    ],
    'End-to-End Pipeline': ['No','No','No','No','Yes'],
    'Decision Support':     ['No','No','No','Partial','Yes'],
    'Interpretable Models': ['No','No','Partial','Yes','Yes'],
}

comp_df = pd.DataFrame(comparison_data)

def highlight_this_study(row):
    if row['Study'] == 'This Study':
        return ['background-color: #dbeafe; font-weight: bold'] * len(row)
    return [''] * len(row)

display(comp_df.style
    .hide(axis='index')
    .apply(highlight_this_study, axis=1)
    .set_caption("Table 4.5 — Comparison with Related Predictive Maintenance Studies")
    .set_table_styles([
        {'selector':'caption', 'props':[('font-weight','bold'),('font-size','13px')]},
        {'selector':'th',      'props':[('background-color','#1F3864'),('color','white'),
                                        ('text-align','center')]},
    ])
)


In [ ]:
# ── RMSE comparison bar chart ─────────────────────────────────────────────────
rmse_studies = {
    'Peng et al.\n(2019) LSTM'      : 12.6,
    'Lee et al.\n(2014) RF'         : 18.3,
    f'This Study\n(Linear Reg.)'    : rmse,
}

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ['#94a3b8', '#94a3b8', '#3b82f6']
bars = ax.bar(rmse_studies.keys(), rmse_studies.values(),
              color=bar_colors, edgecolor='white', linewidth=1.5, width=0.5)

for bar, val in zip(bars, rmse_studies.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('RMSE (cycles)', fontsize=11)
ax.set_title('Figure 4.10 — RMSE Comparison: This Study vs. Related Work\n'
             '(Lower is better; datasets differ — direct comparison is indicative only)',
             fontsize=11, pad=10)
ax.set_ylim(0, 24)
ax.tick_params(axis='x', labelsize=10)

plt.tight_layout()
plt.savefig('fig10_comparison.png', bbox_inches='tight')
plt.show()

print("Note: RMSE values are on different datasets and degradation profiles.")
print("This study's lower RMSE reflects the near-linear simulation context.")


---
## Summary — All Reported Metrics

In [ ]:
print("=" * 60)
print("  CHAPTER 4 — COMPLETE METRICS SUMMARY")
print("=" * 60)
print()
print("  Dataset")
print(f"    Total cycles           : {total_cycles:,}")
print(f"    Processed records      : {processed_rec:,}")
print(f"    Fault rate             : {fault_rate:.2f}%")
print(f"    Feature columns        : {len(FEATURE_COLS_USED)}")
print()
print("  RUL Regressor (Linear Regression)")
print(f"    MSE                    : {mse:.4f}")
print(f"    RMSE                   : {rmse:.4f} cycles")
print(f"    R²                     : {r2:.6f}")
print(f"    Test samples           : {len(y_te_r)}")
print()
print("  Maintenance Classifier (Decision Tree, depth=5)")
print(f"    Accuracy               : {acc:.4f}")
print(f"    Weighted Precision     : {prec:.4f}")
print(f"    Weighted Recall        : {rec:.4f}")
print(f"    Test samples           : {len(y_te_c)}")
print()
print("  Decision Support (full run)")
for level in urg_order:
    cnt = urg_counts.get(level, 0)
    print(f"    {level:<16} : {cnt:5d}  ({cnt/total_dec*100:.1f}%)")
print("=" * 60)
